# From value iteration to PPO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/reinforcement_learning/from_value_iteration_to_ppo.ipynb)

The algorithm ladder of a reinforcement-learning course, climbed on one pendulum and scored on one yardstick. With the model, value iteration solves the Bellman equation on a grid. Without it, the same equation is solved from samples: temporal-difference prediction, exploration, Q-learning, SARSA and Monte Carlo control on the same grid; then the table becomes a network, and the policy itself is trained by REINFORCE, an actor-critic and PPO, with SAC as the off-policy cousin.

Each rung is a minilink object whose source reads like the equation it implements: `DynamicProgrammingPlanner`, `TabularLearningPlanner` (with `QLearning`, `SARSA`, `MonteCarloControl`, `EpsilonGreedy`, `UCB`), `ReinforcementLearningPlanner` (with `REINFORCE`, `ActorCritic`, `PPO`, `SAC`). Every law they return is a controller block scored by `MonteCarloEvaluator`, so the comparison is fair.

| Section | Rung | What replaces what |
| --- | --- | --- |
| 2 | value iteration | the model, swept over every node |
| 3 | temporal-difference prediction | the expectation, replaced by a moving average of samples |
| 4 | exploration | a fixed policy, replaced by one that must choose |
| 5 | Q-learning, SARSA, Monte Carlo control | value iteration, run on observed transitions |
| 6 | function approximation | the table, replaced by a network |
| 7 | REINFORCE, actor-critic, PPO | the value function, replaced by the policy's own gradient |
| 8 | SAC | on-policy batches, replaced by a replay memory |

**Convention.** The course minimizes a cost: stage cost $c_k = g(x_k, u_k)\,\Delta t$, discount $\alpha$, cost-to-go $J^\pi(x)$ and $Q^\pi(x, u)$. The grid learners keep that form. The neural learners follow the reinforcement-learning literature and maximize the reward $r_k = -c_k$ with $\gamma = \alpha$; their learning curves show returns, their Monte Carlo scores the cost.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

## 1. One pendulum, one yardstick

A pendulum with enough torque to lift itself ($12$ N·m against $m g l = 9.81$ N·m), $\theta = 0$ hanging and $\theta = \pi$ upright. The running cost is periodic in the angle, zero upright and two hanging, plus small speed and effort terms. The task is a `StochasticPlanningProblem`: starts drawn anywhere on the circle, an infinite horizon, and an allowed box $|\theta| \le 2\pi$, $|\dot\theta| \le 12$ rad/s whose exit is priced, so every tool below scores the same trajectory the same way. `MonteCarloEvaluator` is the yardstick: the mean cost of a law over 50 random starts, on a 10 s episode.

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from minilink import CostFunction, DynamicProgrammingPlanner, Pendulum
from minilink.control import angle_features
from minilink.planning import (
    DynamicProgrammingOptions,
    EpsilonGreedy,
    MonteCarloEvaluator,
    PolicyEvaluator,
    ReinforcementLearningPlanner,
    StochasticPlanningProblem,
    TabularLearningPlanner,
    UCB,
    Uniform,
)
from minilink.planning.reinforcement_learning import RolloutEnvironment

DT = 0.05  # control period
TORQUE = 12.0  # Nm
ALPHA = 0.98  # discount per control period, a horizon of about 2.5 s
EXIT_COST = 50.0  # price of leaving the box, the same for every tool

plant = Pendulum()
plant.inputs["u"].lower_bound = np.array([-TORQUE])
plant.inputs["u"].upper_bound = np.array([TORQUE])
plant.state.lower_bound = np.array([-2 * np.pi, -12.0])
plant.state.upper_bound = np.array([2 * np.pi, 12.0])


class SwingUpCost(CostFunction):
    def g(self, x, u, t=0.0, params=None):
        theta, dtheta = x
        return (1.0 + jnp.cos(theta))  # traces under JAX, runs on NumPy too + 0.01 * dtheta**2 + 0.001 * u[0] ** 2

    def h(self, x, t=0.0, params=None):
        return 0.0


problem = StochasticPlanningProblem(
    plant,
    cost=SwingUpCost(),
    tf=np.inf,
    x0_distribution=Uniform([-np.pi, -1.0], [np.pi, 1.0]),
        infeasible_cost=EXIT_COST,
    X=plant.state.box,
)
evaluator = MonteCarloEvaluator(problem, dt=DT, n_trials=50, seed=1, backend="numpy")
scores = {}  # every law of this notebook, on the one yardstick
print(problem.horizon_kind(), "horizon | price of infeasibility:", problem.infeasible_cost)

## 2. With the model: value iteration

The grid is the finite world of the whole first half: $41 \times 41$ nodes over the box, three torque levels, one Euler step per control period. Value iteration starts from any table $J$ and repeats the Bellman backup over every node until nothing changes,

$$J(x) \leftarrow \min_u \big[\, g(x, u)\,\Delta t + \alpha\, J\big(x + f(x, u)\,\Delta t\big) \big],$$

the cost-to-go of a successor between nodes being read by linear interpolation. The greedy action of the converged table is the optimal policy $\pi^*$, wrapped as a lookup-table controller block. This is the answer every model-free method below tries to reach without ever calling $f$.

In [ ]:
X_GRID, U_GRID = (41, 41), (3,)

vi = DynamicProgrammingPlanner(problem, x_grid=X_GRID, u_grid=U_GRID, dt=DT, alpha=ALPHA, tol=0.01)
solution = vi.solve()
print(solution.solver)
grid = vi.grid
J_star = vi.result.J
feasible = J_star < EXIT_COST  # nodes from which the box can be kept

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
vi.plot_cost2go(jmax=EXIT_COST, ax=axes[0], title="value iteration: J*", show=False)
vi.plot_policy(ax=axes[1], show=False)
axes[1].set_title("value iteration: greedy torque")
plt.show()

vi_ctl = vi.get_controller()
scores["value iteration"] = evaluator.evaluate(vi_ctl)
print("value iteration law:", scores["value iteration"])

## 3. Without the model, prediction: sampling replaces the expectation

Take the model away. What is left is a plant to interact with: `RolloutEnvironment` wraps the problem as `reset` and `step`, and the learner only sees transitions $(x_k, u_k, c_k, x_{k+1})$. To keep the finite world of the grid, a step goes from node to node: the plant is stepped from the current node and the state it reaches is rounded back to a node.

The first question is *prediction*: the cost-to-go $J^\pi$ of a fixed policy, here the value-iteration law. With the model, policy evaluation is the Bellman backup with the action fixed (`PolicyEvaluator`). Without it, temporal-difference learning waits for one transition and moves the estimate of the current node toward the sample $c_k + \alpha J(x_{k+1})$,

$$J(x_k) \leftarrow J(x_k) + \eta\,\big[\underbrace{c_k + \alpha J(x_{k+1})}_{\text{TD target}} - J(x_k)\big],$$

a moving average of step $\eta$ whose target is itself an estimate (the *bootstrap*). Written by hand below, it is the whole idea of the model-free half in eight lines.

In [ ]:
# Exact policy evaluation of the value-iteration law, with the model
J_pi = PolicyEvaluator(
    problem,
    grid=grid,
    policy=vi_ctl,
    options=DynamicProgrammingOptions(alpha=ALPHA, tol=0.01, out_of_bound_cost=EXIT_COST),
).solve()

# TD(0) from samples: the plant stepped from node to node, the law's action, nothing else
env = RolloutEnvironment(problem, dt=DT, backend="numpy")
rng = np.random.default_rng(0)
J_td = np.zeros(grid.nodes_n)
visited = np.zeros(grid.nodes_n, dtype=bool)
ETA = 0.2
for episode in range(600):
    s = grid.nearest_node(rng.uniform(grid.x_lb, grid.x_ub))  # an exploring start
    x, t = grid.states[s], 0.0
    for k in range(env.n_steps_per_episode):
        x_next, t, reward, terminated, truncated = env.step(x, t, vi_ctl.action(x), rng)
        s_next = grid.nearest_node(x_next)

        # TD(0): J(x) <- J(x) + eta [c + alpha J(x') - J(x)], no value beyond a terminal state
        c = -reward
        J_td[s] += ETA * (c + ALPHA * (1.0 - terminated) * J_td[s_next] - J_td[s])
        visited[s] = True

        if terminated or truncated:
            break
        s, x = s_next, grid.states[s_next]

cells = visited & feasible
print(f"nodes visited: {100 * visited.mean():.0f}%")
print(f"median |J_TD - J_pi| on visited feasible nodes: {np.median(np.abs(J_td - J_pi)[cells]):.2f}, median J_pi: {np.median(J_pi[cells]):.2f}")
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.plot(J_pi[cells], J_td[cells], ".", alpha=0.3)
ax.plot([0, EXIT_COST], [0, EXIT_COST], "k--")
ax.set_xlabel("J_pi, policy evaluation with the model")
ax.set_ylabel("J_TD, temporal differences from samples")
ax.set_xlim(0, 20)
ax.set_ylim(0, 20)
plt.show()

## 4. Exploration: the bandit

Evaluating a policy needs no choice; building one does. The difficulty is isolated by removing the dynamics altogether: one state, $n$ actions, a random cost per pull, the *bandit*. The natural estimate of each action's cost is the moving average of section 3 with $\alpha = 0$, and the question is which action to pull next.

- $\varepsilon$-greedy: a random action with probability $\varepsilon$, otherwise the best known one; $\varepsilon$ usually annealed from 1 toward 0.
- Upper confidence bound: the action minimizing $\hat Q(u) - c\sqrt{\ln N / N(u)}$, an optimistic bonus that fades as an action gets tried.

`EpsilonGreedy` and `UCB` are the exploration objects the grid learners use; a bandit is a grid with one node, so they run here as they are, on the row `Q[s]` and the visit counts `N[s]`. The score of a strategy is its *regret*, the cost paid above the best action.

In [ ]:
arm_costs = np.array([1.0, 0.7, 1.4])  # mean cost of each arm, unknown to the learner
PULLS = 3000

def play(exploration, seed=0):
    rng = np.random.default_rng(seed)
    Q, N = np.zeros(3), np.zeros(3, dtype=int)
    regret = np.zeros(PULLS)
    for k in range(PULLS):
        a = exploration.choose(Q, N, rng, progress=k / PULLS)
        c = arm_costs[a] + rng.normal()
        N[a] += 1
        Q[a] += (c - Q[a]) / N[a]  # sample average: the moving average with step 1/N
        regret[k] = arm_costs[a] - arm_costs.min()
    return np.cumsum(regret), Q

fig, ax = plt.subplots(figsize=(8, 3))
for name, exploration in (
    ("epsilon-greedy, 0.1 fixed", EpsilonGreedy(epsilon=0.1, final=0.1)),
    ("epsilon-greedy, 1 -> 0.01", EpsilonGreedy(epsilon=1.0, final=0.01)),
    ("UCB, c = 1", UCB(c=1.0)),
):
    regret, Q = play(exploration)
    ax.plot(regret, label=name)
    print(f"{name:26s} estimates {np.round(Q, 2)}  regret {regret[-1]:.0f}")
ax.set_xlabel("pull")
ax.set_ylabel("cumulative regret")
ax.legend()
plt.show()

## 5. Control without the model: Q-learning, SARSA, Monte Carlo

Prediction needed $J(x)$; control needs the value of an *action*, $Q(x, u) = g(x, u)\,\Delta t + \alpha J(x')$, because the greedy step $\pi(x) = \arg\min_u Q(x, u)$ then needs no model. Every tabular control method is the moving average of section 3 applied to one cell of the table, and they differ only by the sample $q$ they average:

| Method | Target $q$ | Learns |
| --- | --- | --- |
| Q-learning | $c + \alpha \min_{u'} Q(x', u')$ | $Q^*$, whatever the explorer does next (off-policy) |
| SARSA | $c + \alpha\, Q(x', u')$ with $u'$ the action really taken | the value of the exploring policy itself (on-policy) |
| Monte Carlo control | $\sum_{j \ge k} \alpha^{j-k} c_j$, observed to the episode's end | unbiased, high variance, episodic tasks only |

Q-learning is value iteration run on samples: the same backup, one cell at a time, the expectation over the successor replaced by whatever successor showed up. `TabularLearningPlanner` runs that loop on the value-iteration grid with an exploration object choosing the actions. One detail makes the comparison exact: a successor state between nodes is rounded to a neighbouring node *at random*, with the multilinear weights, so the learner's world is, in expectation, the interpolated grid value iteration sweeps (`rounding="stochastic"`; `"nearest"` aliases a coarse grid). Its result is a cost-to-go field and a greedy policy like value iteration's, so the plots and the controller block are the same.

In [ ]:
EPISODES = 6000  # a few seconds each: one Python step per control period, no model

learners = {}
fig, ax = plt.subplots(figsize=(8, 3))
for name in ("q_learning", "sarsa", "monte_carlo"):
    learner = TabularLearningPlanner(
        problem,
        grid=grid,  # the value-iteration grid, its Euler step, its exit price
        algorithm=name,
        exploration=EpsilonGreedy(epsilon=1.0, final=0.05),
        eta=0.2,
        alpha=ALPHA,
        integrator="euler",
        seed=0,
    )
    solution = learner.solve(episodes=EPISODES)
    learner.plot_learning_curve(ax=ax, window=200)
    learners[name] = learner
    scores[name] = evaluator.evaluate(learner.get_controller())
    error = np.abs(learner.result.J - J_star)[feasible]
    print(f"{name:12s} {solution.solver}: median |J - J*| {np.median(error):.2f} | greedy law {scores[name]}")
ax.legend(["Q-learning", "_", "SARSA", "_", "Monte Carlo", "_"])
ax.set_title("episode cost while learning, exploration included")
ax.set_ylim(0, 60)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
vi.plot_cost2go(jmax=EXIT_COST, ax=axes[0], title="value iteration: J*", show=False)
learners["q_learning"].plot_cost2go(jmax=EXIT_COST, ax=axes[1], title="Q-learning: min_u Q", show=False)
plt.show()

The Q-learning table follows $J^*$ where the exploring episodes went, noisier near the priced edge of the box, and its greedy law swings up like the value-iteration law at a somewhat higher cost: a moving average of random targets keeps a noise of order $\eta$. SARSA learns the value of the *exploring* policy: safer while it explores, a little more conservative once it stops (the cliff demo `demos/rl/double_integrator_sarsa_vs_q_learning_rl.py` makes that difference visible). Monte Carlo control averages whole-episode costs and needs many more episodes to see through their variance.

Two more knobs of the same loop are worth turning: `exploration=UCB(c=...)` replaces the random exploration by an optimistic bonus, and `eta=None` uses the sample average $1/N(x, u)$ instead of a constant step.

## 6. Beyond the table: function approximation

A table is an approximator with one indicator basis function per cell: $Q(x, u) = \sum_{i} w_i\, \mathbf{1}_i(x, u)$, and the moving average is stochastic gradient descent on the squared error to the target. Replacing the indicators by a parameterized function $\hat Q(x, u \mid w)$ keeps the same targets and the same descent, $w \leftarrow w + \eta\,\delta_k\,\nabla_w \hat Q$, and lets the value generalize between samples, at the price of the convergence guarantees. The table below could not have been much finer: its size grows as the product of the levels per axis. A small network on periodic features of the state has fewer parameters than the grid has cells and no box to leave.

That network is exactly what the second half of the ladder trains: the critics `ValueFunction` and `QFunction`, and the policy itself, `NeuralPolicyController`, a controller block whose weights are its `params`.

In [ ]:
from minilink.control import NeuralPolicyController

features = angle_features(angles=[0], scales={1: 0.1})  # cos theta, sin theta, scaled rate
policy_block = NeuralPolicyController(plant, features=features, hidden=(32, 32))
n_weights = sum(w.size for w in policy_block.params["mlp"].values())
print(f"grid cells: {grid.nodes_n * grid.actions_n}   network weights: {n_weights}")

## 7. Policy gradient: REINFORCE, actor-critic, PPO

The last family trains the policy directly. A stochastic policy $\pi_\theta(u \mid x)$, a Gaussian around the network's mean torque, makes the expected cost a smooth function of $\theta$, and the policy gradient theorem gives its gradient from samples alone,

$$\nabla_\theta J(\theta) = \mathbb{E}\big[\nabla_\theta \ln \pi_\theta(u \mid x)\, Q^\pi(x, u)\big].$$

The three rungs differ in what stands in for $Q^\pi$ and in how far one update may go:

- **REINFORCE**: the cost observed to the end of the episode, minus the batch mean as a baseline; no critic, unbiased, noisy.
- **Actor-critic**: a learned critic $V_w(x)$; its temporal-difference errors, combined by generalized advantage estimation, are the advantage the actor follows; one gradient step per batch.
- **PPO**: the same, with the probability ratio $\pi_\theta / \pi_{\text{old}}$ clipped so a step stays in a trust region, and several passes over each batch.

`ReinforcementLearningPlanner` runs the three with one loop, one stochastic policy, one environment; each rung gets the budget it needs. The learning curves show the exploration return, the scores the same yardstick as the tables above. The mathematics of each piece, checked line by line, is in `policy_gradient_to_ppo.ipynb`.

In [ ]:
ladder = {
    "REINFORCE": (dict(algorithm="reinforce", n_envs=64, learning_rate=1e-2, episode_length=4.0), 800_000),
    "actor-critic": (dict(algorithm="actor_critic", n_envs=16, n_steps=32, learning_rate=3e-3), 300_000),
    "PPO": (dict(algorithm="ppo", n_envs=64, n_steps=32, batch_size=256, learning_rate=3e-3), 100_000),
}
planners = {}
fig, ax = plt.subplots(figsize=(8, 3))
for name, (settings, budget) in ladder.items():
    planner = ReinforcementLearningPlanner(
        problem, dt=DT, features=features, hidden=(32, 32), gamma=ALPHA, **settings
    )
    solution = planner.solve(timesteps=budget)
    planner.plot_learning_curve(ax=ax)
    planners[name] = planner
    scores[name] = evaluator.evaluate(planner.get_controller())
    print(f"{name:13s} {solution.solver}")
ax.legend(list(ladder))
ax.set_title("mean episode return while learning (reward = -cost)")
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, planner) in zip(axes, planners.items()):
    planner.get_controller().plot_control_law(x_axis=0, y_axis=1, u_axis=0, ax=ax, show=False)
    ax.set_title(f"{name}: torque")
plt.show()

## 8. Off-policy with a replay memory: SAC

The three rungs above learn from the batch their current policy just collected, then throw it away. Because Q-learning's target does not care which policy produced a transition, an action-value critic can learn from *any* stored experience: Soft Actor-Critic keeps every transition in a replay memory, draws minibatches from it, trains twin critics $Q_w(x, u)$ on the soft Bellman target and the actor by the reparameterized gradient through them, and adapts a temperature that keeps the policy exploring. Far fewer plant steps, far more gradient steps: the trade between the two families.

In [ ]:
sac = ReinforcementLearningPlanner(
    problem, dt=DT, features=features, hidden=(32, 32), gamma=ALPHA,
    algorithm="sac", n_envs=8, n_steps=16, learning_starts=2000,
)
solution = sac.solve(timesteps=20_000)
scores["SAC"] = evaluator.evaluate(sac.get_controller())
print(solution.solver)

## 9. One yardstick

Every law of this notebook, tables and networks alike, is a controller block; `MonteCarloEvaluator` closed the loop with each on the same 50 starts. The closed loop with the PPO law, simulated on the continuous plant, ends the tour.

In [ ]:
print(f"{'law':18s} {'mean cost':>10s} {'worst':>8s} {'failures':>9s}")
for name, report in scores.items():
    print(f"{name:18s} {report.mean:10.2f} {report.worst:8.2f} {100 * report.failure_rate:8.0f}%")

plant.x0 = np.array([0.05, 0.0])  # hanging, a tiny tip
cl_sys = planners["PPO"].get_controller() @ plant
cl_sys.name = "Pendulum with the PPO law"
traj = cl_sys.compute_trajectory(tf=6.0, dt=0.01, verbose=False)
cl_sys.plot_trajectory(traj)
cl_sys.animate(traj)

## Summary

- Value iteration solves the Bellman equation on a grid by sweeping every node with the model; every model-free method solves the same equation from observed transitions.
- Temporal-difference learning replaces the expectation by a moving average of bootstrapped samples; Q-values make the greedy step model-free; exploration decides which samples are seen.
- Q-learning, SARSA and Monte Carlo control are one loop with three targets: the best next action, the action taken, the cost observed to the end.
- A network replaces the table when the grid gets too large; the same targets train it by gradient descent.
- Policy-gradient methods train the law itself: REINFORCE with Monte Carlo returns, an actor-critic with a learned baseline, PPO with a trust region; SAC learns off-policy from a replay memory.

**What to read next:** `policy_gradient_to_ppo.ipynb` for the mathematics of section 7, `11_reinforcement_learning.ipynb` for the planner API and the learned law as a `System`, `demos/rl/` for the cliff, the underactuated swing-ups, the car and the rocket.